# Importing Libraries

In [277]:
# Data manipulation and numerical operations
import pandas as pd   # For data manipulation and analysis
import numpy as np    # For numerical operations

#import shap  # package used to calculate Shap values

# Data visualization libraries
import matplotlib.pyplot as plt   # For plotting and visualizations
import graphviz   # For visualizing decision trees
import seaborn as sns
import plotly.express as px

# Scikit-learn: Model selection and evaluation
from sklearn.model_selection import train_test_split   # For splitting the data into training and testing sets
from sklearn.metrics import accuracy_score, classification_report  # For evaluating model performance
from sklearn.metrics import roc_curve, roc_auc_score, auc  # For calculating ROC curve and AUC
from sklearn.metrics import confusion_matrix

# Scikit-learn: Preprocessing
from sklearn.preprocessing import LabelEncoder   # For encoding categorical labels into numeric values
from sklearn.preprocessing import OneHotEncoder  # For encoding categorical features using one-hot encoding
from sklearn.preprocessing import StandardScaler

# Scikit-learn: Machine learning model
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
#from catboost import CatBoostClassifier

# Scikit-learn: Tree visualization
from sklearn.tree import export_graphviz  # For exporting decision trees in DOT format
from sklearn.tree import plot_tree  # For plotting decision trees

from scipy import stats

plt.style.use('fivethirtyeight')
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

from imblearn.over_sampling import SMOTE

import joblib

# Parameters

In [278]:
USE_SMOTE = True
USE_RATIO_PERCENTAGE = False

SHOW_CLASSIFICATION_REPORT = False
SHOW_ROC_CURVE = False
SHOW_CONFUSION_MATRIX = False
SHOW_FEATURE_IMPORTANCE = False

# 1.1 Loading the Data into a Pandas DataFrame

In [ ]:
# Specify the file path to the CSV file containing the thyroid cancer recurrence data
csv_path = "../data/thyroid_diff.csv"

# Read the CSV file into a DataFrame using pandas
df = pd.read_csv(csv_path)

# Display the first few rows of the DataFrame to get a quick overview of the data
# df.head()

# Box plot of Age by Gender
plt.figure(figsize=(10, 6))
sns.boxplot(x='Recurred', y='Age', data=df)
plt.title('Box Plot of Age by Recurred')
plt.xlabel('Recurred')
plt.ylabel('Age')
plt.show()

# Box plot of Age by Stage
plt.figure(figsize=(10, 6))
sns.boxplot(x='Stage', y='Age', data=df)
plt.title('Box Plot of Age by Stage')
plt.xlabel('Stage')
plt.ylabel('Age')
plt.show()

# Gender by Risk
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='Gender', hue='Risk', palette='viridis')
plt.title('Grouped Bar Chart of Gender vs Risk')
plt.show()

# Scatter plot of Age vs. Tumor size (T)
fig = px.scatter(df, x='Age', y='T', title='Scatter Plot of Age vs. Tumor size (T)', labels={'T': 'Tumor size (T)', 'Age': 'Age'})

# Show the plot
fig.show()

### 1.1.1 Label Encoding

In [280]:
# Label encoding yes or no values

# Define mapping
mapping = {'No': 0, 'Yes': 1}

# TODO: Insert names of columns that have Yes or No values
boolean_columns = ['Smoking','Hx Smoking','Hx Radiotherapy','Recurred']
df[boolean_columns] = df[boolean_columns].map(mapping.get)

# df.head()

In [281]:
# Label encode features with only 2 values
le = LabelEncoder()

# TODO: Input column names that only have 2 values
columns_to_encode = ['Gender','Focality','M']

for column in columns_to_encode:
  df[column] = le.fit_transform(df[column])

# df.head()

### 1.1.2 Label Encoding with Order

In [282]:
# TODO: Define mapping for 'Physical Examination'
mapping = {'Normal': 0, 'Diffuse goiter':1, 'Single nodular goiter-left':2, 'Single nodular goiter-right':2,'Multinodular goiter':3}

df['Physical Examination'] = df['Physical Examination'].map(mapping)

if np.isnan(df['Physical Examination'].unique()).any():
  raise ValueError('Your output has NaN values. Check your mapping inputs to make sure they exactly match the unique values.')

# df.head()

In [283]:
# TODO: Define mapping for 'Adenopathy'
mapping = {'No': 0, 'Right':1, 'Left':1, 'Posterior':2,'Bilateral':3,'Extensive':4}

df['Adenopathy'] = df['Adenopathy'].map(mapping)

if np.isnan(df['Adenopathy'].unique()).any():
  raise ValueError('Your output has NaN values. Check your mapping inputs to make sure they exactly match the unique values.')

# df.head()

In [284]:
# TODO: Define mapping for 'Pathology'
mapping = {'Micropapillary':0,'Papillary':1, 'Follicular':2, 'Hurthle cell':3}

df['Pathology'] = df['Pathology'].map(mapping)

if np.isnan(df['Pathology'].unique()).any():
  raise ValueError('Your output has NaN values. Check your mapping inputs to make sure they exactly match the unique values.')

# df.head()

In [285]:
# TODO: Define mapping for 'Risk'
mapping = {'Low':0,'Intermediate':1, 'High':2}

df['Risk'] = df['Risk'].map(mapping)

if np.isnan(df['Risk'].unique()).any():
  raise ValueError('Your output has NaN values. Check your mapping inputs to make sure they exactly match the unique values.')

# df.head()

In [286]:
# TODO: Define mapping for 'T'
mapping = {'T1a':0, 'T1b':1, 'T2':2, 'T3a':3, 'T3b':4, 'T4a':5, 'T4b':6}

df['T'] = df['T'].map(mapping)

if np.isnan(df['T'].unique()).any():
  raise ValueError('Your output has NaN values. Check your mapping inputs to make sure they exactly match the unique values.')

# df.head()

In [287]:
# TODO: Define mapping for 'N'
mapping = {'N0':0, 'N1a':1, 'N1b':2}

df['N'] = df['N'].map(mapping)

if np.isnan(df['N'].unique()).any():
  raise ValueError('Your output has NaN values. Check your mapping inputs to make sure they exactly match the unique values.')

# df.head()

In [288]:
# TODO: Define mapping for 'Stage'
mapping = {'I':0, 'II':1, 'III':2, 'IVA':3, 'IVB':4}

df['Stage'] = df['Stage'].map(mapping)

if np.isnan(df['Stage'].unique()).any():
  raise ValueError('Your output has NaN values. Check your mapping inputs to make sure they exactly match the unique values.')

# df.head()

In [289]:
# TODO: Define mapping for 'Response'
mapping = {'Excellent':0, 'Indeterminate':1, 'Biochemical Incomplete':2, 'Structural Incomplete':3}

df['Response'] = df['Response'].map(mapping)

if np.isnan(df['Response'].unique()).any():
  raise ValueError('Your output has NaN values. Check your mapping inputs to make sure they exactly match the unique values.')

# df.head()

### 1.1.3 One-Hot Encoding

In [ ]:
# Initialize a OneHotEncoder object
ohe = OneHotEncoder()

# Fit the OneHotEncoder to the 'Thyroid Function' column of the dataframe 'df'
# and transform it into a one-hot encoded array. The 'toarray()' method
# converts the sparse matrix output into a dense numpy array.
ohe.fit_transform(df[['Thyroid Function']]).toarray()

In [ ]:
# Fit the OneHotEncoder to the 'Thyroid Function' column of the dataframe 'df'
# and transform it into a one-hot encoded array. The 'toarray()' method
# converts the sparse matrix output into a dense numpy array.
feature_array = ohe.fit_transform(df[['Thyroid Function']]).toarray()

# Get the category labels from the fitted OneHotEncoder
feature_labels = ohe.categories_

# Convert the list of arrays (from ohe.categories_) to a flattened numpy array
feature_labels = np.array(feature_labels).ravel()

# Create a DataFrame from the one-hot encoded array with column names set to the flattened feature labels
pd.DataFrame(feature_array, columns=feature_labels)

In [ ]:
# Create a DataFrame from the one-hot encoded array (feature_array)
# using the flattened feature labels (feature_labels) as column names
features = pd.DataFrame(feature_array, columns=feature_labels)

# Concatenate the original dataframe (df) and the new one-hot encoded features (features)
# along the columns (axis=1). This results in a combined DataFrame with both the original
# and one-hot encoded features.
pd.concat([df, features], axis=1)

In [293]:
# Concatenate the original dataframe (df) with the new one-hot encoded features (features)
# along the columns (axis=1). This adds the one-hot encoded columns to the original dataframe.
df = pd.concat([df, features], axis=1)

# Drop the original 'Thyroid Function' column from the dataframe
# as it has been replaced by the one-hot encoded columns.
df = df.drop(['Thyroid Function'], axis='columns')

# Display the first few rows of the updated dataframe to inspect the result.
# df.head()

In [294]:
#apply Box-Cox transformation to 'Age' column
age_boxcox, _ = stats.boxcox(df['Age'])

#apply Min-Max scaling to the transformed 'Age' column
scaler = MinMaxScaler()
age_scaled = scaler.fit_transform(age_boxcox.reshape(-1, 1)).flatten()

#replace 'Age' column with the scaled values
df['Age'] = age_scaled

# df.head()

# Choose Features

In [295]:
#---------------------------------------------------------
# Choose Clinico-Pathological Features
#---------------------------------------------------------
df = df[['Age', 'Gender', 'Response', 'Recurred']]


### 1.1.4 Split to Train and Test

In [ ]:
# Drop the 'Recurred' column from the DataFrame to create the input features DataFrame
inputs = df.drop('Recurred', axis='columns')

# Select the target column 'Recurred' from the DataFrame to create the target DataFrame
target = df['Recurred']

if USE_RATIO_PERCENTAGE:
    numeric_cols = inputs.select_dtypes(include=[np.number]).columns.tolist()
   
    #---------------------------------------------------------
    # Ratio-based Feature Engineering
    # Numeric columns: Age, Gender, Stage
    #---------------------------------------------------------
    def safe_div(x, y):
        return x / (y + 1e-9)

    if 'Age' in numeric_cols and 'Gender' in numeric_cols:
        inputs['Age_Gender_ratio'] = safe_div(inputs['Age'], inputs['Gender'])
        inputs['Gender_Age_ratio'] = safe_div(inputs['Gender'], inputs['Age'])

    if 'Stage' in numeric_cols and 'Age' in numeric_cols:
        inputs['PhysExam_Age_ratio'] = safe_div(inputs['Stage'], inputs['Age'])
        inputs['Age_PhysExam_ratio'] = safe_div(inputs['Age'], inputs['Stage'])

    #---------------------------------------------------------
    # Final Detailed Evaluation with Ratio Features
    #---------------------------------------------------------
    X_final = inputs.copy()
    scaler_final = StandardScaler()
    X_final_scaled = scaler_final.fit_transform(X_final)
    
    # Split the data into train and test sets (80% train, 20% test)
    X_train, X_test, y_train, y_test = train_test_split(X_final_scaled, target, test_size=0.2, random_state=42)
else:
    X_train, X_test, y_train, y_test = train_test_split(inputs, target, test_size=0.2, random_state=42)

# print("X_train shape:", X_train.shape)
# print("X_test shape:", X_test.shape)
# print("y_train shape:", y_train.shape)
# print("y_test shape:", y_test.shape)

if USE_SMOTE:
    #---------------------------------------------------------
    # Apply SMOTE to balance the class distribution
    #---------------------------------------------------------
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

    # print("Before SMOTE - Class distribution in y_train:")
    # print(y_train.value_counts())
    # print("After SMOTE - Class distribution in y_train_resampled:")
    # print(y_train_resampled.value_counts())

    # print("X_train_resampled shape:", X_train_resampled.shape)
    # print("y_train_resampled shape:", y_train_resampled.shape)

inputs.head()

1.2 Train the Model

In [297]:
#instantiate classification models
models = {
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(),
    'K-Nearest Neighbors': KNeighborsClassifier(),
    'Logistic Regression': LogisticRegression(),
    'Support Vector Machine': SVC(),
    'Gaussian Naive Bayes': GaussianNB(),
    'Gradient Boosting': GradientBoostingClassifier(),
    'AdaBoost': AdaBoostClassifier(),
    'XGBoost': XGBClassifier() #,'CatBoost': CatBoostClassifier(verbose=0),
}

model_scores = []
for name, model in models.items():
    if USE_SMOTE:
       model.fit(X_train_resampled, y_train_resampled)
    else:
       model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    accuracy = round(accuracy_score(y_test, y_pred), 2)
    
    if hasattr(model, "predict_proba"):
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_pred_proba = [0] * len(y_test)  
   
    model_scores.append((name, y_test, y_pred, y_pred_proba, model, accuracy))

## Evaluate Model

In [ ]:
sm = sorted(model_scores , key=lambda x :x[5] ,reverse = True)

for model in sm:
    # Calculate ROC Curve and AUC
    fpr, tpr, _ = roc_curve(model[1], model[3])
    roc_auc = auc(fpr, tpr) * 100
    model_accuracy = accuracy_score(model[1], model[2]) * 100
    cls_report = classification_report(model[1], model[2])

    print(f'Model = {model[0]} has Accuracy = {model_accuracy} and AUC = {roc_auc}')
  
    if SHOW_CLASSIFICATION_REPORT:
        # Print Classification Report
        print(f'Classification Report : \n{cls_report}\n')

    if SHOW_ROC_CURVE:
        # Plot ROC Curve
        plt.figure(figsize=(10, 7))
        plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.2f})")
        plt.plot([0, 1], [0, 1], color="gray", linestyle="--")  # Diagonal line for reference
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("ROC Curve for Classification Models")
        plt.legend(loc="lower right")
        plt.show()

    if SHOW_CONFUSION_MATRIX:
        # Plot Confusion Matrix
        plt.figure(figsize=(10,7))
        sns.heatmap(confusion_matrix(model[1], model[2]) , annot=True , fmt='d')
        plt.xlabel('Prediction')
        plt.ylabel('Actual')
        plt.show()

    if SHOW_FEATURE_IMPORTANCE:
        # Plot Feature importance
        if hasattr(model[4], "feature_importances_"):
            feature_names = list(df.columns)
            importances = model[4].feature_importances_
            indices = np.argsort(importances)

            # Plot feature importance
            plt.figure(figsize=(10, 7))
            plt.title('Feature Importances')
            plt.barh(range(len(indices)), importances[indices], color='b', align='center')
            plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
            plt.xlabel('Relative Importance')
            plt.show()

In [ ]:
for model in sm:
    # Calculate ROC Curve and AUC
    fpr, tpr, _ = roc_curve(model[1], model[3])
    roc_auc = auc(fpr, tpr) * 100
    model_accuracy = accuracy_score(model[1], model[2]) * 100
    cls_report = classification_report(model[1], model[2])

    print(f'Model = {model[0]} has Accuracy = {model_accuracy} and AUC = {roc_auc}')

    # Select row 253 from the inputs DataFrame
    specific_row = df.iloc[253]

    # Reshape the row to match the expected input shape for the model
    specific_row_reshaped = specific_row.drop('Recurred').values.reshape(1, -1)

    # Make a prediction on the selected row
    prediction = model[4].predict(specific_row_reshaped)

    # Retrieve the actual value from the 'recurred' column for row 253
    actual_value = specific_row['Recurred']

    # Print the prediction and the actual value
    print(f"Prediction: {prediction[0]}")
    print(f"Actual Value: {actual_value}")
    #print(specific_row)
  
    if SHOW_CLASSIFICATION_REPORT:
        # Print Classification Report
        print(f'Classification Report : \n{cls_report}\n')

    if SHOW_ROC_CURVE:
        # Plot ROC Curve
        plt.figure(figsize=(10, 7))
        plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.2f})")
        plt.plot([0, 1], [0, 1], color="gray", linestyle="--")  # Diagonal line for reference
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title("ROC Curve for Classification Models")
        plt.legend(loc="lower right")
        plt.show()

    if SHOW_CONFUSION_MATRIX:
        # Plot Confusion Matrix
        plt.figure(figsize=(10,7))
        sns.heatmap(confusion_matrix(model[1], model[2]) , annot=True , fmt='d')
        plt.xlabel('Prediction')
        plt.ylabel('Actual')
        plt.show()

    if SHOW_FEATURE_IMPORTANCE:
        # Plot Feature importance
        if hasattr(model[4], "feature_importances_"):
            feature_names = list(df.columns)
            importances = model[4].feature_importances_
            indices = np.argsort(importances)

            # Plot feature importance
            plt.figure(figsize=(10, 7))
            plt.title('Feature Importances')
            plt.barh(range(len(indices)), importances[indices], color='b', align='center')
            plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
            plt.xlabel('Relative Importance')
            plt.show()
     
    joblib.dump(model[4], '../model/pjas-thyroid-{0}.pkl'.format(model[0]))